# 01 — Préparation des données

**Rôle de ce notebook** : construire, à partir des tables brutes (`train`, `test`, `Parcellaire`,
`Apports N.P.K.`, `Irrigation`, `BD_Climat_Domaines`), une table maître au grain *une ligne =
une parcelle × un mois de récolte* (identique à `train.csv`/`test.csv`), enrichie des variables
structurelles, d'irrigation, de fertilisation et de climat.

Sortie : `Data/processed/train_master.csv` et `Data/processed/test_master.csv`.

## 1. Chargement des tables brutes

In [1]:

import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

DATA_DIR = Path(r"C:\Github_Projects\Competition_Kaggle\Kaggle_competition\Data")
OUT_DIR = DATA_DIR / "processed"

FULL_KEY = ['Campagne', 'Region', 'Code Ferme', 'Libelle Site', 'Libelle Variete', 'Num Parcelle']

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
parcellaire = pd.read_csv(DATA_DIR / "Parcellaire.csv")
npk = pd.read_csv(DATA_DIR / "Apports N.P.K.csv")
irrigation = pd.concat(
    [pd.read_csv(DATA_DIR / f"Irrigation{suf}.csv") for suf in ["", "_2", "_3", "_4", "_5"]],
    ignore_index=True,
)
irrigation = irrigation.rename(columns={"Code Traçabilité": "CodeTracabilite"})

print(f"train: {train.shape}, test: {test.shape}, parcellaire: {parcellaire.shape}")
print(f"npk: {npk.shape}, irrigation (concat des 5 fichiers): {irrigation.shape}")


train: (1629, 10), test: (411, 10), parcellaire: (1375, 14)
npk: (58056, 11), irrigation (concat des 5 fichiers): (269615, 9)


## 2. Valeurs aberrantes de saisie

Quelques valeurs d'irrigation et d'apports NPK sont manifestement erronées (facteurs 10 à 1000
par rapport aux valeurs voisines) et une parcelle a des surfaces nulles. Chaque correction est
appliquée par appariement `Num Parcelle + Date`, avec contrôle de la valeur d'origine.

In [2]:

for c in ("Num Parcelle",):
    npk[c] = npk[c].astype(str)
    irrigation[c] = irrigation[c].astype(str)
    parcellaire[c] = parcellaire[c].astype(str)
npk["Date"] = pd.to_datetime(npk["Date"])
irrigation["Date"] = pd.to_datetime(irrigation["Date"])

POINT_FIXES_IRRIGATION = [
# (Num Parcelle, Date, valeur_attendue, valeur_corrigee)
    ("20106", "2021-08-23", 182474.249305, 297.0),
]
POINT_FIXES_NPK = [
# (Num Parcelle, Date, valeur_attendue, valeur_corrigee)
    ("15010", "2024-09-30", "K", 2154.99975, 215.0),
    ("30040", "2025-02-28", "P", 1370.7, 300.0),
    ("30030", "2025-03-01", "P", 1171.2, 596.0),
    ("30080", "2025-02-27", "P", 1179.3, 596.0),
]

for num_parcelle, date, avant, apres in POINT_FIXES_IRRIGATION:
    mask = (irrigation["Num Parcelle"] == num_parcelle) & (irrigation["Date"] == pd.Timestamp(date))
    n_match = mask.sum()
    assert n_match == 1, f"Correction irrigation {num_parcelle}/{date}: {n_match} ligne(s) trouvée(s), attendu 1"
    val = irrigation.loc[mask, "Qté eau (m3)"].iloc[0]
    assert abs(val - avant) < 1, f"Valeur inattendue avant correction: {val} != {avant}"
    irrigation.loc[mask, "Qté eau (m3)"] = apres
    print(f"Irrigation {num_parcelle} / {date}: {avant:.0f} -> {apres:.0f} m3")

for num_parcelle, date, col, avant, apres in POINT_FIXES_NPK:
    mask = (npk["Num Parcelle"] == num_parcelle) & (npk["Date"] == pd.Timestamp(date))
    n_match = mask.sum()
    assert n_match == 1, f"Correction NPK {num_parcelle}/{date}: {n_match} ligne(s) trouvée(s), attendu 1"
    val = npk.loc[mask, col].iloc[0]
    assert abs(val - avant) < 1, f"Valeur inattendue avant correction: {val} != {avant}"
    npk.loc[mask, col] = apres
    print(f"NPK {num_parcelle} / {date} / {col}: {avant:.0f} -> {apres:.0f} kg")

# Parcelle à exclure : Sup Debut Camp = Sup Fin Camp = 0 (10040, OULED GNAOU, campagne 22/23)
excl_mask = (
    (parcellaire["Num Parcelle"] == "10040")
    & (parcellaire["Campagne"] == "22/23")
    & (parcellaire["Sup Debut Camp"] == 0)
    & (parcellaire["Sup Fin Camp"] == 0)
)
print(f"\nParcelle(s) exclue(s) de Parcellaire (surfaces nulles des deux côtés) : {excl_mask.sum()}")
parcellaire = parcellaire[~excl_mask].reset_index(drop=True)


Irrigation 20106 / 2021-08-23: 182474 -> 297 m3
NPK 15010 / 2024-09-30 / K: 2155 -> 215 kg
NPK 30040 / 2025-02-28 / P: 1371 -> 300 kg
NPK 30030 / 2025-03-01 / P: 1171 -> 596 kg
NPK 30080 / 2025-02-27 / P: 1179 -> 596 kg

Parcelle(s) exclue(s) de Parcellaire (surfaces nulles des deux côtés) : 1


## 3. Harmonisation des clés et des types

Uniformisation des types, de la casse des régions (`Souss` / `SOUSS`) et du libellé `Ferme 16`
(équivalent de `Site 16`). `CodeTracabilite` regroupe plusieurs parcelles : il n'est pas utilisé
comme clé, la clé composite `Campagne + Region + Code Ferme + Libelle Site + Libelle Variete +
Num Parcelle` lui est substituée.

In [3]:

def normalize_keys(df):
    df = df.copy()
    if "Num Parcelle" in df.columns:
        df["Num Parcelle"] = df["Num Parcelle"].astype(str).str.strip()
    for c in ["Region", "Code Ferme", "Libelle Site", "Libelle Variete", "Campagne", "CodeTracabilite"]:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()
    if "Region" in df.columns:
        df["Region"] = df["Region"].str.upper()
    if "Libelle Site" in df.columns:
        df["Libelle Site"] = df["Libelle Site"].replace({"Ferme 16": "Site 16"})
    return df

train = normalize_keys(train)
test = normalize_keys(test)
parcellaire = normalize_keys(parcellaire)
npk = normalize_keys(npk)
irrigation = normalize_keys(irrigation)

dup_full_key = train.duplicated(subset=FULL_KEY + ["Date"]).sum()
dup_code = train.duplicated(subset=["CodeTracabilite", "Date"]).sum()
print(f"Doublons sur la clé composite complète + Date (train) : {dup_full_key} (attendu : 0)")
print(f"Doublons sur CodeTracabilite + Date (train) : {dup_code}")
print("-> CodeTracabilite seul est une cle non fiable (il regroupe plusieurs Num Parcelle) : "
      "la clé composite est utilisée pour toutes les jointures.")


Doublons sur la clé composite complète + Date (train) : 0 (attendu : 0)
Doublons sur CodeTracabilite + Date (train) : 418
-> CodeTracabilite seul est une cle non fiable (il regroupe plusieurs Num Parcelle) : la clé composite est utilisée pour toutes les jointures.


## 4. Nettoyage de `Parcellaire`

Les quelques doublons sur la clé composite (parcelle scindée en deux lignes) sont regroupés en
additionnant les surfaces. La surface retenue est `Sup Fin Camp` si elle est > 0, sinon
`Sup Debut Camp` (parcelle arrachée en cours de campagne).

In [4]:

parcellaire["Date Plantation"] = pd.to_datetime(parcellaire["Date Plantation"], errors="coerce")
parcellaire["Date Arrachage"] = pd.to_datetime(parcellaire["Date Arrachage"], errors="coerce")

before = len(parcellaire)
agg_map = {
    "CodeTracabilite": "first",
    "Date Plantation": "min",
    "Date Arrachage": "max",
    "Porte Greffe": "first",
    "Sup Debut Camp": "sum",
    "Sup Plantee": "sum",
    "Sup Arrachee": "sum",
    "Sup Fin Camp": "sum",
}
parcellaire = parcellaire.groupby(FULL_KEY, as_index=False).agg(agg_map)
print(f"Parcellaire : {before} -> {len(parcellaire)} lignes après regroupement des vrais doublons.")

# Logique officielle (GUIDE_FINAL_NETTOYAGE_BD_KAGGLE.docx) :
# une parcelle arrachée en cours de campagne a Sup Fin Camp = 0 mais a bien produit
# (Sup Debut Camp > 0) -> on prend Sup Fin Camp si > 0, sinon Sup Debut Camp.
parcellaire["Surface"] = np.where(
    parcellaire["Sup Fin Camp"] > 0, parcellaire["Sup Fin Camp"], parcellaire["Sup Debut Camp"]
)

print(f"Lignes Parcellaire avec Region manquante (pépinières/tunnels type 'TUNNEL PORTE GREFFE') : "
      f"{parcellaire['Region'].isna().sum()}")

parcellaire_cols = FULL_KEY + ["CodeTracabilite", "Date Plantation", "Porte Greffe", "Surface"]
parcellaire_slim = parcellaire[parcellaire_cols]


Parcellaire : 1374 -> 1370 lignes après regroupement des vrais doublons.
Lignes Parcellaire avec Region manquante (pépinières/tunnels type 'TUNNEL PORTE GREFFE') : 0


## 5. Fusion `Parcellaire` → `train` / `test`

Jointure sur la clé composite, avec assertion que le nombre de lignes reste inchangé.

In [5]:

merge_cols = ["Date Plantation", "Porte Greffe", "Surface"]
train_m = train.merge(parcellaire_slim[FULL_KEY + merge_cols], on=FULL_KEY, how="left")
test_m = test.merge(parcellaire_slim[FULL_KEY + merge_cols], on=FULL_KEY, how="left")

print(f"train : {train.shape} -> {train_m.shape}")
print(f"test  : {test.shape} -> {test_m.shape}")
assert len(train_m) == len(train), "EXPLOSION DE JOINTURE DETECTEE SUR TRAIN"
assert len(test_m) == len(test), "EXPLOSION DE JOINTURE DETECTEE SUR TEST"

missing_struct_train = train_m["Surface"].isna().sum()
print(f"\nLignes train sans info Parcellaire pour leur campagne (ex. TR11600) : {missing_struct_train}")
train_m.loc[train_m["Surface"].isna(), FULL_KEY + ["CodeTracabilite"]]


train : (1629, 10) -> (1629, 13)
test  : (411, 10) -> (411, 13)

Lignes train sans info Parcellaire pour leur campagne (ex. TR11600) : 4


,Campagne,Region,Code Ferme,Libelle Site,Libelle Variete,Num Parcelle,CodeTracabilite
203,20/21,ORIENTAL,Ferme 13,Site 13,Variete4,11043,TR11600
384,21/22,GHARB,Ferme 3,Site 3,Variete2,13010,TR11120
385,21/22,GHARB,Ferme 3,Site 3,Variete2,13010,TR11120
386,21/22,GHARB,Ferme 3,Site 3,Variete2,13010,TR11120


## 6. Imputation hiérarchique des informations structurelles manquantes

Repli en cascade : (1) même parcelle sur une autre campagne, (2) médiane Région × Variété.
Un flag `parcelle_info_manquante` conserve l'information sur les lignes concernées.

In [6]:

def impute_structural(df, parcellaire_slim):
    df = df.copy()
    df["parcelle_info_manquante"] = df["Surface"].isna().astype(int)
    key_no_camp = [c for c in FULL_KEY if c != "Campagne"]

    fallback1 = (
        parcellaire_slim.groupby(key_no_camp)[["Surface", "Porte Greffe", "Date Plantation"]]
        .agg({"Surface": "median",
              "Porte Greffe": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
              "Date Plantation": "min"})
        .reset_index()
    )
    df = df.merge(fallback1, on=key_no_camp, how="left", suffixes=("", "_fb1"))
    for c in ["Surface", "Porte Greffe", "Date Plantation"]:
        df[c] = df[c].fillna(df[f"{c}_fb1"])
        df.drop(columns=[f"{c}_fb1"], inplace=True)

    fallback2 = (
        parcellaire_slim.groupby(["Region", "Libelle Variete"])[["Surface"]]
        .median().reset_index().rename(columns={"Surface": "Surface_fb2"})
    )
    df = df.merge(fallback2, on=["Region", "Libelle Variete"], how="left")
    df["Surface"] = df["Surface"].fillna(df["Surface_fb2"])
    df.drop(columns=["Surface_fb2"], inplace=True)
    df["Porte Greffe"] = df["Porte Greffe"].fillna("Inconnu")
    return df

train_m = impute_structural(train_m, parcellaire_slim)
test_m = impute_structural(test_m, parcellaire_slim)

print(f"Train - lignes flaggées 'parcelle_info_manquante' : {train_m['parcelle_info_manquante'].sum()}")
print(f"Test  - lignes flaggées 'parcelle_info_manquante' : {test_m['parcelle_info_manquante'].sum()}")
print(f"Surface encore manquante après repli - train : {train_m['Surface'].isna().sum()}, "
      f"test : {test_m['Surface'].isna().sum()}")


Train - lignes flaggées 'parcelle_info_manquante' : 4
Test  - lignes flaggées 'parcelle_info_manquante' : 0
Surface encore manquante après repli - train : 0, test : 0


## 7. Âge au moment de la récolte

L'âge est calculé par rapport à la date de récolte de chaque ligne. Un âge négatif (récolte
antérieure à la date de plantation enregistrée, typiquement une parcelle replantée) est mis à
`NaN` et signalé par le flag `age_incoherente`.

In [7]:

for df in (train_m, test_m):
    df["Date"] = pd.to_datetime(df["Date"])
    df["Age_recolte"] = (df["Date"] - df["Date Plantation"]).dt.days / 365.25
    df["year"] = df["Date"].dt.year
    df["month"] = df["Date"].dt.month
    df["age_incoherente"] = (df["Age_recolte"] < 0).astype(int)
    df.loc[df["Age_recolte"] < 0, "Age_recolte"] = np.nan

print(f"Âges incohérents détectés - train : {train_m['age_incoherente'].sum()}, "
      f"test : {test_m['age_incoherente'].sum()}")
train_m["Age_recolte"].describe()


Âges incohérents détectés - train : 3, test : 0


count    1625.000000
mean       17.570827
std        17.685021
min         1.519507
25%         5.412731
50%        13.423682
75%        18.666667
max        81.248460
Name: Age_recolte, dtype: float64

## 8. Nettoyage NPK et agrégation mensuelle

Les apports d'azote négatifs sont ramenés à 0. Irrigation et NPK sont agrégés au mois par
parcelle avant toute jointure.

In [8]:

neg_n_sites = npk.loc[npk["N"] < 0, "Libelle Site"].unique()
neg_n = (npk["N"] < 0).sum()
print(f"Valeurs N négatives dans NPK : {neg_n} (sites concernés : {list(neg_n_sites)}) -> ramenées à 0")
npk["N"] = npk["N"].clip(lower=0)
npk["P"] = npk["P"].clip(lower=0)
npk["K"] = npk["K"].clip(lower=0)

npk["month_idx"] = npk["Date"].dt.year * 12 + npk["Date"].dt.month
irrigation["month_idx"] = irrigation["Date"].dt.year * 12 + irrigation["Date"].dt.month

npk_monthly = npk.groupby(FULL_KEY + ["month_idx"], as_index=False)[["N", "P", "K"]].sum()
irr_monthly = irrigation.groupby(FULL_KEY + ["month_idx"], as_index=False)["Qté eau (m3)"].sum()

print(f"NPK mensuel : {npk_monthly.shape}, Irrigation mensuelle : {irr_monthly.shape}")


Valeurs N négatives dans NPK : 8 (sites concernés : ['Site 12']) -> ramenées à 0


NPK mensuel : (9444, 10), Irrigation mensuelle : (13524, 8)


## 9. Cumuls glissants (1 / 3 / 6 / 12 mois avant la récolte)

Pour chaque ligne, irrigation et apports N, P, K sont cumulés sur les 1, 3, 6 et 12 mois
précédant la date de récolte, afin de refléter le décalage entre l'apport et son effet sur le
fruit (phénologie décrite dans le guide).

In [9]:

monthly_inputs = irr_monthly.merge(npk_monthly, on=FULL_KEY + ["month_idx"], how="outer")
for c in ["Qté eau (m3)", "N", "P", "K"]:
    monthly_inputs[c] = monthly_inputs[c].fillna(0.0)

full_range = np.arange(monthly_inputs["month_idx"].min(), monthly_inputs["month_idx"].max() + 1)

def build_rolling_table(monthly_df, keys):
    out = []
    value_cols = [c for c in monthly_df.columns if c not in keys + ["month_idx"]]
    for key_vals, g in monthly_df.groupby(keys):
        g = g.set_index("month_idx").reindex(full_range, fill_value=0.0)
        cum = g[value_cols].cumsum()
        res = pd.DataFrame(index=full_range)
        for w in (1, 3, 6, 12):
            shifted = cum.shift(w).fillna(0.0)
            res_w = cum - shifted
            res_w.columns = [f"{c}_cum_{w}m" for c in value_cols]
            res = pd.concat([res, res_w], axis=1)
        res["month_idx"] = full_range
        for k, v in zip(keys, key_vals if isinstance(key_vals, tuple) else (key_vals,)):
            res[k] = v
        out.append(res)
    return pd.concat(out, ignore_index=True)

rolling_features = build_rolling_table(monthly_inputs, FULL_KEY)
print(f"Table de cumuls glissants : {rolling_features.shape}")

train_m["month_idx"] = train_m["year"] * 12 + train_m["month"]
test_m["month_idx"] = test_m["year"] * 12 + test_m["month"]
train_m = train_m.merge(rolling_features, on=FULL_KEY + ["month_idx"], how="left")
test_m = test_m.merge(rolling_features, on=FULL_KEY + ["month_idx"], how="left")

print(f"train : {train_m.shape} (doit rester {len(train)} lignes)")
print(f"test  : {test_m.shape} (doit rester {len(test)} lignes)")
assert len(train_m) == len(train)
assert len(test_m) == len(test)


Table de cumuls glissants : (76680, 23)
train : (1629, 35) (doit rester 1629 lignes)
test  : (411, 35) (doit rester 411 lignes)


### Corrélation avec le tonnage

Comparaison des cumuls sur 1 mois et sur 3 mois avec `log(1 + Tonnage)`.

In [10]:

train_m["log_tonnage"] = np.log1p(train_m["Tonnage (Kg)"])
print("Corrélation avec log(Tonnage) - cumul 1 mois :")
print(train_m[["Qté eau (m3)_cum_1m", "N_cum_1m", "log_tonnage"]].corr()["log_tonnage"].drop("log_tonnage"))
print("\nCorrélation avec log(Tonnage) - cumul 3 mois glissants :")
print(train_m[["Qté eau (m3)_cum_3m", "N_cum_3m", "log_tonnage"]].corr()["log_tonnage"].drop("log_tonnage"))


Corrélation avec log(Tonnage) - cumul 1 mois :
Qté eau (m3)_cum_1m    0.159046
N_cum_1m               0.212993
Name: log_tonnage, dtype: float64

Corrélation avec log(Tonnage) - cumul 3 mois glissants :
Qté eau (m3)_cum_3m    0.255637
N_cum_3m               0.217202
Name: log_tonnage, dtype: float64


## 10. Intégration du climat au niveau site

Le mapping `Libelle Site → nom_domaine` (`Data/site_domain_mapping.csv`) permet de joindre le
climat au niveau de chaque site. Pour les sites sans station propre, on utilise la station de
repli indiquée dans la feuille *Instruction_Données manquantes* du fichier climat, et à défaut la
moyenne régionale.

In [11]:

site_map = pd.read_csv(DATA_DIR / "site_domain_mapping.csv")
site_map[["Libelle Site", "nom_domaine", "domaine_climat_effectif", "fallback_utilise"]]


,Libelle Site,nom_domaine,domaine_climat_effectif,fallback_utilise
0,Site 1,AIT OUALLAL,AIT OUALLAL,non
1,Site 2,DAR ESSALAM,DAR ESSALAM,non
2,Site 3,JORF I,JORF I,non
3,Site 4,JORF II,JORF I,oui
4,Site 5,SIDI MOUSSA,NaN,oui
5,Site 6,OUARGHA,OUARGHA,non
6,Site 7,AGAFAI,AGAFAI,non
7,Site 8,AGDAL,AGDAL,non
8,Site 9,EL BAHJA I,EL BAHJA I,non
9,Site 10,EL BAHJA II,EL BAHJA I,oui


In [12]:

climat = pd.read_excel(DATA_DIR / "BD_Climat_Domaines.xlsx", sheet_name="BD_Climat")
climat["Date"] = pd.to_datetime(climat["Date"])
climat["month_idx"] = climat["Date"].dt.year * 12 + climat["Date"].dt.month
num_cols = ["TemperatureMax C°", "TemperatureMin C°", "TemperatureMoy C°",
            "HumiditeRelativeMax %", "HumiditeRelativeMin %", "HumiditeRelativeMoy %", "ET0(mm)"]
for c in num_cols:
    climat[c] = pd.to_numeric(climat[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")

climat["Region_norm"] = climat["Region"].astype(str).str.strip().str.upper()
region_monthly = climat.groupby(["Region_norm", "month_idx"], as_index=False)[num_cols].mean()
region_monthly = region_monthly.rename(columns={c: f"{c}_region" for c in num_cols})
domain_monthly = climat.groupby(["nom_domaine", "month_idx"], as_index=False)[num_cols].mean()

site_to_domain = site_map.set_index("Libelle Site")["domaine_climat_effectif"].to_dict()

def attach_climate(df):
    df = df.copy()
    df["domaine_climat_effectif"] = df["Libelle Site"].map(site_to_domain)
    df = df.merge(domain_monthly, left_on=["domaine_climat_effectif", "month_idx"],
                  right_on=["nom_domaine", "month_idx"], how="left")
    df = df.merge(region_monthly, left_on=["Region", "month_idx"], right_on=["Region_norm", "month_idx"],
                  how="left")
    df["climat_source"] = np.where(df[num_cols].notna().all(axis=1), "site", "region_repli")
    for c in num_cols:
        df[c] = df[c].fillna(df[f"{c}_region"])
    df = df.drop(columns=["nom_domaine", "Region_norm"] + [f"{c}_region" for c in num_cols], errors="ignore")
    return df

before_tr, before_te = len(train_m), len(test_m)
train_m = attach_climate(train_m)
test_m = attach_climate(test_m)
assert len(train_m) == before_tr and len(test_m) == before_te

print("Source du climat utilisée :")
print("train:", dict(train_m["climat_source"].value_counts()))
print("test:", dict(test_m["climat_source"].value_counts()))
print("\nSites en repli régional (pas de station propre ni de repli fourni) :",
      sorted(train_m.loc[train_m['climat_source'] == 'region_repli', 'Libelle Site'].unique()))


Source du climat utilisée :
train: {'site': np.int64(1622), 'region_repli': np.int64(7)}
test: {'site': np.int64(411)}

Sites en repli régional (pas de station propre ni de repli fourni) : ['Site 5']


## 11. Climat en moyenne glissante (3 / 6 mois avant récolte)

Température, humidité et ET0 moyennes sur les 3 et 6 mois précédant la récolte.

In [13]:

def rolling_mean_table(monthly_df, key_col, value_cols, windows=(3, 6)):
    out = []
    full_idx = np.arange(monthly_df["month_idx"].min(), monthly_df["month_idx"].max() + 1)
    for key_val, g in monthly_df.groupby(key_col):
        g = g.set_index("month_idx").reindex(full_idx)
        res = pd.DataFrame(index=full_idx)
        for w in windows:
            roll = g[value_cols].rolling(window=w, min_periods=1).mean()
            roll.columns = [f"{c}_roll{w}m" for c in value_cols]
            res = pd.concat([res, roll], axis=1)
        res["month_idx"] = full_idx
        res[key_col] = key_val
        out.append(res)
    return pd.concat(out, ignore_index=True)

domain_roll = rolling_mean_table(domain_monthly, "nom_domaine", num_cols)
region_roll = rolling_mean_table(region_monthly, "Region_norm", [f"{c}_region" for c in num_cols])

def attach_climate_rolling(df):
    df = df.copy()
    df = df.merge(domain_roll, left_on=["domaine_climat_effectif", "month_idx"],
                  right_on=["nom_domaine", "month_idx"], how="left")
    df = df.merge(region_roll, left_on=["Region", "month_idx"], right_on=["Region_norm", "month_idx"],
                  how="left")
    roll_cols_domain = [c for c in domain_roll.columns if c.endswith("m")]
    for c in roll_cols_domain:
        region_equiv = c.replace(c.split("_roll")[0], c.split("_roll")[0] + "_region")
        if region_equiv in df.columns:
            df[c] = df[c].fillna(df[region_equiv])
    df = df.drop(columns=["nom_domaine", "Region_norm"], errors="ignore")
    df = df.drop(columns=[c for c in df.columns if c.endswith("m") and "_region_" in c], errors="ignore")
    return df

before_tr, before_te = len(train_m), len(test_m)
train_m = attach_climate_rolling(train_m)
test_m = attach_climate_rolling(test_m)
assert len(train_m) == before_tr and len(test_m) == before_te
print("OK - climat glissant intégré, aucune ligne perdue/dupliquée.")


OK - climat glissant intégré, aucune ligne perdue/dupliquée.


## 12. Rendement N-1 (alternance)

Le guide identifie l'historique de production (année N-1) comme un des facteurs les plus explicatifs (capture l'alternance ON/OFF). On calcule le tonnage total de la campagne précédente pour la même parcelle physique.

In [14]:

campagne_rank = {c: i for i, c in enumerate(sorted(set(train["Campagne"]).union(test["Campagne"])))}

tonnage_campagne = (
    train_m.groupby(FULL_KEY[1:] + ["Campagne"], as_index=False)["Tonnage (Kg)"].sum()
    .rename(columns={"Tonnage (Kg)": "tonnage_campagne_totale"})
)
tonnage_campagne["campagne_rank"] = tonnage_campagne["Campagne"].map(campagne_rank)
prev = tonnage_campagne.copy()
prev["campagne_rank"] += 1
prev = prev.rename(columns={"tonnage_campagne_totale": "rendement_N-1"}).drop(columns=["Campagne"])

def attach_prev_yield(df):
    df = df.copy()
    df["campagne_rank"] = df["Campagne"].map(campagne_rank)
    df = df.merge(prev, on=FULL_KEY[1:] + ["campagne_rank"], how="left")
    df["rendement_N-1_manquant"] = df["rendement_N-1"].isna().astype(int)
    return df.drop(columns=["campagne_rank"])

before_tr, before_te = len(train_m), len(test_m)
train_m = attach_prev_yield(train_m)
test_m = attach_prev_yield(test_m)
assert len(train_m) == before_tr and len(test_m) == before_te

print(f"Rendement N-1 disponible - train : {(1 - train_m['rendement_N-1_manquant']).sum()}/{len(train_m)}, "
      f"test : {(1 - test_m['rendement_N-1_manquant']).sum()}/{len(test_m)}")
print("(NaN normal pour la première campagne 20/21, ou une parcelle sans historique l'année précédente)")


Rendement N-1 disponible - train : 1122/1629, test : 366/411
(NaN normal pour la première campagne 20/21, ou une parcelle sans historique l'année précédente)


## 13. Contrôle des bornes de rendement

Rendement annuel par parcelle-campagne : moyenne attendue ≈ 17,6 T/ha, maximum réaliste
~60-100 T/ha, seuil dur < 220 T/ha.

In [15]:

annual = train_m.groupby(FULL_KEY, as_index=False).agg({"Tonnage (Kg)": "sum", "Surface": "first"})
annual["Rendement_T_ha"] = annual["Tonnage (Kg)"] / (annual["Surface"] * 1000)
print(annual["Rendement_T_ha"].describe())
n_over_220 = (annual["Rendement_T_ha"] > 220).sum()
n_over_100 = (annual["Rendement_T_ha"] > 100).sum()
print(f"\nParcelles-campagnes avec Rendement > 220 T/ha (seuil dur) : {n_over_220}")
print(f"Parcelles-campagnes avec Rendement > 100 T/ha (règle d'or agronomique) : {n_over_100}")
if n_over_100 > 0:
    display(annual.loc[annual["Rendement_T_ha"] > 100].sort_values("Rendement_T_ha", ascending=False))


count    794.000000
mean      17.584911
std       15.322466
min        0.007407
25%        5.887211
50%       15.579272
75%       25.788912
max      216.177784
Name: Rendement_T_ha, dtype: float64

Parcelles-campagnes avec Rendement > 220 T/ha (seuil dur) : 0
Parcelles-campagnes avec Rendement > 100 T/ha (règle d'or agronomique) : 1


,Campagne,Region,Code Ferme,Libelle Site,Libelle Variete,Num Parcelle,Tonnage (Kg),Surface,Rendement_T_ha
69,20/21,HAOUZ,Ferme 12,Site 12,Variete22,21135,24320.0,0.1125,216.177784


## 14. Assemblage final, renommage propre et sauvegarde

In [16]:

RENAME = {
    "Qté eau (m3)": "Irrigation_m3",
    "TemperatureMax C°": "TemperatureMax_C",
    "TemperatureMin C°": "TemperatureMin_C",
    "TemperatureMoy C°": "TemperatureMoy_C",
    "HumiditeRelativeMax %": "HumiditeRelativeMax_pct",
    "HumiditeRelativeMin %": "HumiditeRelativeMin_pct",
    "HumiditeRelativeMoy %": "HumiditeRelativeMoy_pct",
    "ET0(mm)": "ET0_mm",
}
def apply_rename(df):
    new_cols = {}
    for c in df.columns:
        base = c
        for old, new in RENAME.items():
            if c.startswith(old):
                base = c.replace(old, new)
                break
        new_cols[c] = base
    return df.rename(columns=new_cols)

train_m = apply_rename(train_m)
test_m = apply_rename(test_m)

print(f"train_master final : {train_m.shape}")
print(f"test_master final  : {test_m.shape}")
assert len(train_m) == len(train)
assert len(test_m) == len(test)

miss = train_m.isna().sum()
print("\nValeurs manquantes restantes (train) :")
miss[miss > 0].sort_values(ascending=False)


train_master final : (1629, 61)
test_master final  : (411, 60)

Valeurs manquantes restantes (train) :


rendement_N-1              507
N_cum_1m                    46
P_cum_1m                    46
Irrigation_m3_cum_1m        46
N_cum_6m                    46
P_cum_6m                    46
Irrigation_m3_cum_3m        46
K_cum_1m                    46
P_cum_3m                    46
K_cum_3m                    46
Irrigation_m3_cum_6m        46
N_cum_3m                    46
N_cum_12m                   46
P_cum_12m                   46
Irrigation_m3_cum_12m       46
K_cum_6m                    46
K_cum_12m                   46
domaine_climat_effectif      7
Age_recolte                  4
Date Plantation              1
dtype: int64

In [17]:

check_cols = [
    "Age_recolte", "Surface", "rendement_N-1",
    "Irrigation_m3_cum_3m", "Irrigation_m3_cum_12m",
    "N_cum_3m", "N_cum_12m", "K_cum_12m",
    "TemperatureMoy_C_roll3m", "ET0_mm_roll3m",
]
check_cols = [c for c in check_cols if c in train_m.columns]
print("Corrélation finale avec log(Tonnage) — principales features :")
train_m[check_cols + ["log_tonnage"]].corr()["log_tonnage"].drop("log_tonnage").sort_values(ascending=False)


Corrélation finale avec log(Tonnage) — principales features :


rendement_N-1              0.353897
Surface                    0.342838
Irrigation_m3_cum_3m       0.255637
K_cum_12m                  0.250417
Irrigation_m3_cum_12m      0.241762
N_cum_3m                   0.217202
Age_recolte                0.213662
N_cum_12m                  0.182885
TemperatureMoy_C_roll3m   -0.173836
ET0_mm_roll3m             -0.178683
Name: log_tonnage, dtype: float64

In [18]:

train_m.to_csv(OUT_DIR / "train_master.csv", index=False)
test_m.to_csv(OUT_DIR / "test_master.csv", index=False)
print(f"Sauvegardé : {OUT_DIR / 'train_master.csv'}  {train_m.shape}")
print(f"Sauvegardé : {OUT_DIR / 'test_master.csv'}   {test_m.shape}")
print("\n=== Table maître sauvegardée ===")


Sauvegardé : C:\Github_Projects\Competition_Kaggle\Kaggle_competition\Data\processed\train_master.csv  (1629, 61)
Sauvegardé : C:\Github_Projects\Competition_Kaggle\Kaggle_competition\Data\processed\test_master.csv   (411, 60)

=== Table maître sauvegardée ===
